# 18 — Expanded ensemble voters

This notebook presents the independently specified third-voter screen. It combines persisted, aligned out-of-fold probabilities and performs no model refit. The local test remains closed.

**Result:** none of the CatBoost, LightGBM or MLP additions improves the accepted two-voter ensemble. The closest is a 5% MLP contribution at 81.601% accuracy, 0.023 percentage points below baseline.

## Course-aligned lifecycle

| Step | Application |
| --- | --- |
| 1. Define the goal and scope | Test materially different third voters around the accepted ensemble. |
| 2. Gather the data | Reuse persisted development-fold OOF probabilities. |
| 3. Explore the data | Compare strength, disagreement, unique correct rows and probability correlation. |
| 4. Clean and preprocess the data | Not applicable to the no-refit screen; cached voters already use fold-safe preprocessing. |
| 5. Select and engineer features | Preserve each voter's established representation. |
| 6. Define the machine-learning task | Three-class nominal classification. |
| 7. Partition the data | Reuse the untouched local test and five frozen development folds. |
| 8. Select and train candidate methods | Combine three predeclared voters at fixed low weights; no training is required. |
| 9. Evaluate and interpret the results | Apply the established gate and inspect accuracy, recall and probability quality. |
| 10. Deploy and iterate | Not applicable: no candidate passes the gate. |

In [ ]:
from pathlib import Path
import sys
import joblib

STAGE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = STAGE_DIR / 'src'
PROJECT_DIR = STAGE_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

result = joblib.load(
    PROJECT_DIR / '.runtime' / 'expanded-ensemble-screen'
    / 'expanded-ensemble-screen.joblib'
)
result.candidate_summary

## Interpretation

The candidates preserve the accepted 55:45 XGBoost-to-Random-Forest ratio inside the remaining probability mass. CatBoost tests a native-categorical representation, bagged LightGBM tests the strongest earlier tree complement, and the MLP tests the most diverse plausible voter.

All three slightly improve log loss and Brier score while reducing accuracy. LightGBM is strong but highly correlated with the accepted probabilities. The MLP is more diverse, but the accepted ensemble is uniquely correct on almost twice as many rows as the MLP. More voters therefore add probability smoothing without enough new correct argmax decisions.

The full reasoning and stop decision are in [`../reports/expanded-ensemble-voter-screen.md`](../reports/expanded-ensemble-voter-screen.md).

In [ ]:
result.component_summary.sort_values(
    'baseline_disagreement',
    ascending=False,
)